# Tutorial 3: Mouse Brain (RNA + ATAC)

This tutorial demonstrates the complete ProtoGlue workflow on the **Mouse Brain** dataset (RNA + ATAC spatial multi-omics).

## Steps
1. Environment setup
2. Data loading & preprocessing
3. Graph construction
4. Model initialization & pretraining
5. Automatic K selection
6. Fine-tuning with DEC clustering
7. Evaluation & visualization

## 1. Environment Setup

In [ ]:
import torch, os
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
import protoglue as pg
import protoglue.config as cfg
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

print('ProtoGlue version:', pg.__version__)

## 2. Configuration

In [ ]:
DATA_DIR = Path('data/mouse_brain')
OUT_DIR  = Path('output/mouse_brain')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 2022
pg.fix_seed(SEED)

print(f'Device: {DEVICE}')
print(f'Data dir: {DATA_DIR}')
print(f'Output dir: {OUT_DIR}')

## 3. Data Loading & Preprocessing

Load the Mouse Brain dataset (RNA + ATAC) and perform standard preprocessing:
- RNA: HVG selection -> normalization -> log1p -> scale -> PCA
- ATAC: CLR normalization -> scale -> PCA

In [ ]:
pca1, pca2, coords, adata_rna, adata_om2 = pg.load_and_preprocess(
    data_dir=DATA_DIR,
    rna_file='adata_RNA.h5ad',
    om2_file='adata_peaks_normalized.h5ad',
    n_hvg=cfg.N_HVG,
    n_pca=cfg.N_PCA,
    seed=SEED,
    verbose=True,
)

print(f'RNA PCA shape: {pca1.shape}')
print(f'ATAC PCA shape: {pca2.shape}')
print(f'Spatial coords shape: {coords.shape}')
print(f'Spots: {adata_rna.n_obs}')

## 4. Graph Construction

Build multi-scale spatial graphs (density-adaptive + fixed KNN) and feature-space graphs.

In [ ]:
A_sp_list, A_f1, A_f2 = pg.build_graphs(
    coords=coords, pca1=pca1, pca2=pca2,
    use_amsg=cfg.USE_AMSG,
    k_base=cfg.K_BASE, k_min=cfg.K_MIN, k_max=cfg.K_MAX,
    density_alpha=cfg.DENSITY_ALPHA,
    spatial_scales=cfg.SPATIAL_SCALES,
    feat_k=cfg.FEAT_K,
)
print(f'Spatial graphs: {len(A_sp_list)} scales')
for i, A in enumerate(A_sp_list):
    print(f'  Scale {i}: {A.nnz} edges')

# Convert to torch sparse tensors
dev = torch.device(DEVICE)
A_sp_t, A_f1_t, A_f2_t = pg.graphs_to_torch(A_sp_list, A_f1, A_f2, dev)
X1_t = torch.tensor(pca1, dtype=torch.float32, device=dev)
X2_t = torch.tensor(pca2, dtype=torch.float32, device=dev)
print('Tensors moved to', DEVICE)

## 5. Model Initialization & Pretraining

In [ ]:
model = pg.ProtoGlue(
    in1=X1_t.shape[1], in2=X2_t.shape[1], n_scales=len(A_sp_t),
).to(dev)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

# Calibrate smoothness weight
from protoglue.losses.smooth import calibrate_lam_smooth
cfg.LAM_SMOOTH = calibrate_lam_smooth(model, X1_t, X2_t, A_sp_t, A_f1_t, A_f2_t)
print(f'Calibrated LAM_SMOOTH = {cfg.LAM_SMOOTH}')

In [ ]:
from protoglue.losses.total import total_loss

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=40, min_lr=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=cfg.USE_AMP and 'cuda' in DEVICE)

pretrain_logs = pg.run_stage(
    stage='pretrain', epochs=cfg.PRETRAIN_EPOCHS,
    model=model, X1_t=X1_t, X2_t=X2_t,
    A_sp_t=A_sp_t, A_f1_t=A_f1_t, A_f2_t=A_f2_t,
    optimizer=optimizer, scheduler=scheduler, scaler=scaler,
    loss_fn=total_loss, output_dir=OUT_DIR, ckpt_tag='pretrain',
    WARMUP_EPOCHS=cfg.WARMUP_EPOCHS, PATIENCE=cfg.PATIENCE,
    GRAD_CLIP=cfg.GRAD_CLIP, SEED=SEED,
)
print(f'Pretrain finished: {len(pretrain_logs)} epochs')

## 6. Automatic K Selection

Scan candidate K values with composite scoring.

In [ ]:
ckpt = torch.load(OUT_DIR / 'best_pretrain.pt', map_location=dev, weights_only=True)
model.load_state_dict(ckpt['model'])

model.eval()
with torch.no_grad():
    z0 = model(X1_t, X2_t, A_sp_t, A_f1_t, A_f2_t)['z'].cpu().numpy()

gt = None  # No ground truth available for this dataset

best_k, kscan_df = pg.scan_select_k(z=z0, A_sp0=A_sp_list[0], gt=gt, K_GRID=cfg.K_GRID, SEED=SEED)
print(f'Selected K = {best_k}')
print(kscan_df[['K', 'silhouette', 'spatial_edge_purity', 'score_select']].to_string(index=False))

## 7. Fine-tuning with DEC Clustering

In [ ]:
from sklearn.cluster import KMeans

dec_head = pg.DECHead(best_k, cfg.FUSION_DIM).to(dev)
km = KMeans(n_clusters=best_k, random_state=SEED, n_init=50).fit(z0)
with torch.no_grad():
    dec_head.mu.copy_(torch.tensor(km.cluster_centers_, dtype=torch.float32, device=dev))

optimizer_ft = torch.optim.AdamW(
    list(model.parameters()) + list(dec_head.parameters()),
    lr=cfg.LR * 0.5, weight_decay=cfg.WEIGHT_DECAY,
)
scheduler_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_ft, mode='min', factor=0.5, patience=40, min_lr=1e-6)
scaler_ft = torch.amp.GradScaler('cuda', enabled=cfg.USE_AMP and 'cuda' in DEVICE)

finetune_logs = pg.run_stage(
    stage='finetune', epochs=cfg.FINETUNE_EPOCHS,
    model=model, X1_t=X1_t, X2_t=X2_t,
    A_sp_t=A_sp_t, A_f1_t=A_f1_t, A_f2_t=A_f2_t,
    dec_head=dec_head,
    optimizer=optimizer_ft, scheduler=scheduler_ft, scaler=scaler_ft,
    loss_fn=total_loss, warm_offset=cfg.PRETRAIN_EPOCHS,
    output_dir=OUT_DIR, ckpt_tag='finetune',
    WARMUP_EPOCHS=cfg.WARMUP_EPOCHS, PATIENCE=cfg.PATIENCE,
    GRAD_CLIP=cfg.GRAD_CLIP, DEC_UPDATE_INTERVAL=cfg.DEC_UPDATE_INTERVAL,
    P_SMOOTH=cfg.P_SMOOTH, COLLAPSE_CHECK_INTERVAL=cfg.COLLAPSE_CHECK_INTERVAL,
    MIN_CLUSTER_FRAC=cfg.MIN_CLUSTER_FRAC, SEED=SEED,
    TEMP_ANNEAL_EPOCHS=cfg.TEMP_ANNEAL_EPOCHS,
    TEMP_START=cfg.TEMP_START, TEMP_END=cfg.TEMP_END,
)
print(f'Fine-tuning finished: {len(finetune_logs)} epochs')

## 8. Evaluation

In [ ]:
ckpt = torch.load(OUT_DIR / 'best_finetune.pt', map_location=dev, weights_only=True)
model.load_state_dict(ckpt['model'])
if ckpt['dec_head'] is not None:
    dec_head.load_state_dict(ckpt['dec_head'])

model.eval()
with torch.no_grad():
    out_final = model(X1_t, X2_t, A_sp_t, A_f1_t, A_f2_t, dec_head=dec_head)

z = out_final['z'].cpu().numpy()
q = out_final['q'].cpu().numpy()
labels = q.argmax(axis=1).astype(int)

metrics = pg.eval_labels(labels, A_sp0=A_sp_list[0], gt=gt, z=z, x1=pca1, x2=pca2)
print('Evaluation metrics:')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

## 9. Visualization

In [ ]:
from protoglue.plotting.colors import cluster_cmap

# Flip y for proper spatial orientation
xy_plot = coords.copy()
xy_plot[:, 1] = -xy_plot[:, 1]
colors = cluster_cmap(best_k)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Spatial domains
ax = axes[0]
for k in range(best_k):
    mask = labels == k
    ax.scatter(xy_plot[mask, 0], xy_plot[mask, 1], c=colors[k], s=3, alpha=0.8, label=str(k))
ax.set_title('Spatial Domains')
ax.set_aspect('equal')
ax.legend(markerscale=3, fontsize=6, ncol=2)

# UMAP
ax = axes[1]
try:
    import umap
    umap_xy = umap.UMAP(n_components=2, random_state=SEED).fit_transform(z)
    for k in range(best_k):
        mask = labels == k
        ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1], c=colors[k], s=3, alpha=0.8)
    ax.set_title('UMAP (by cluster)')
except ImportError:
    ax.text(0.5, 0.5, 'pip install umap-learn\nfor UMAP visualization', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('UMAP (umap-learn not installed)')

# Modality weight
ax = axes[2]
w_mod = out_final['alpha_modality'][:, 0].cpu().numpy()
sc_ = ax.scatter(xy_plot[:, 0], xy_plot[:, 1], c=w_mod, cmap='RdBu_r', s=3, alpha=0.8)
ax.set_title('RNA Weight (alpha)')
ax.set_aspect('equal')
plt.colorbar(sc_, ax=ax, shrink=0.6)

plt.tight_layout()
plt.savefig(OUT_DIR / 'overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to', OUT_DIR / 'overview.png')

## 10. Save Results

In [ ]:
adata_rna.obs['protoglue_labels'] = labels.astype(str)
adata_rna.obsm['protoglue_latent'] = z

from protoglue.data.io import safe_write_h5ad
safe_write_h5ad(adata_rna, OUT_DIR / 'adata_result.h5ad')
print('Results saved to', OUT_DIR / 'adata_result.h5ad')

---
## Alternative: One-Line Pipeline

All steps above can be run with a single function call:

```python
result = pg.run_pipeline(
    data_dir='data/mouse_brain',
    output_dir='output/mouse_brain',
)
labels = result['labels']
metrics = result['metrics']
```